In [1]:
!git clone https://github.com/Brayan695/RNAseq-AMD.git

Cloning into 'RNAseq-AMD'...
remote: Enumerating objects: 2460, done.
remote: Counting objects: 100% (480/480), done.
remote: Compressing objects: 100% (351/351), done.
remote: Total 2460 (delta 217), reused 369 (delta 122), pack-reused 1980 (from 1)
Receiving objects: 100% (2460/2460), 1.03 GiB | 20.36 MiB/s, done.
Resolving deltas: 100% (876/876), done.
Updating files: 100% (1724/1724), done.


In [23]:
import warnings
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.feature_selection import f_classif
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [24]:
DATA_PATH = "/kaggle/working/RNAseq-AMD/Dataset/MetaSheet_1_4.csv"
N_ITERATIONS = 1000
SAMPLE_FRACTION = 0.8
TOP_K_FRACTION = 0.30
RANDOM_SEED = 2026

EXCLUDE_LABEL_LEAKAGE = True
LEAKAGE_COLUMNS = [
    "oc_AMD",
    "oc_dry AMD",
    "oc_macular degeneration",
    "oc_Wet AMD",
    "oc_wet AMD",
    "oc_early AMD",
    "oc_possible AMD",
    "oc_possible macular degeneration",
    "oc_AMD (received shots)",
]

rng = np.random.default_rng(RANDOM_SEED)

In [25]:
def load_data(path):
    """Load metadata, split into features (X) and label (y).
    y = 1 for late AMD (mgs_level == 4), 0 for control (mgs_level == 1).
    """
    df = pd.read_csv(path)
    y = (df["mgs_level"] == 4).astype(int).values
    X = df.drop(columns=["sample_id", "mgs_level"])
    return X, y

In [27]:
X_raw, y = load_data(DATA_PATH)
print(f"  {X_raw.shape[0]} samples, {X_raw.shape[1]} raw features")
print(f"  class balance: {sum(y==0)} control, {sum(y==1)} late AMD")

  166 samples, 613 raw features
  class balance: 105 control, 61 late AMD


In [28]:
def drop_constant_features(X):
    """Remove features with zero variance in this cohort -- they carry
    no information and break ANOVA/Kruskal-Wallis (denominator = 0)."""
    nunique = X.nunique()
    constant_cols = nunique[nunique <= 1].index.tolist()
    X_filtered = X.drop(columns=constant_cols)
    return X_filtered, constant_cols


In [31]:
if EXCLUDE_LABEL_LEAKAGE:
        present = [c for c in LEAKAGE_COLUMNS if c in X_raw.columns]
        X_raw = X_raw.drop(columns=present)
        print(f"  dropped {len(present)} label-leakage columns (chart notes "
              f"that restate the AMD diagnosis): {present}")

X, dropped = drop_constant_features(X_raw)
print(f"  dropped {len(dropped)} constant (zero-variance) features")
print(f"  {X.shape[1]} usable features remain")

top_k = max(1, int(X.shape[1] * TOP_K_FRACTION))
top_n = max(1, int(X.shape[1] * TOP_N_FRACTION))
print(f"  per-iteration top-K = {top_k}, final top-N per method = {top_n}")

  dropped 9 label-leakage columns (chart notes that restate the AMD diagnosis): ['oc_AMD', 'oc_dry AMD', 'oc_macular degeneration', 'oc_Wet AMD', 'oc_wet AMD', 'oc_early AMD', 'oc_possible AMD', 'oc_possible macular degeneration', 'oc_AMD (received shots)']
  dropped 314 constant (zero-variance) features
  290 usable features remain
  per-iteration top-K = 87, final top-N per method = 29


In [32]:
def anova_scores(X, y):
    """ANOVA F-test score per feature, all features at once
    (sklearn's f_classif is the vectorized equivalent of scipy.stats.f_oneway)."""
    f_stat, _ = f_classif(X.values, y)
    f_stat = np.nan_to_num(f_stat, nan=0.0)
    return pd.Series(f_stat, index=X.columns)


def auc_scores(X, y):
    """AUC per feature, computed directly from ranks (equivalent to
    roc_auc_score but vectorized across every column at once):
        AUC = (sum of ranks in the positive class - n1*(n1+1)/2) / (n1*n0)
    Take max(auc, 1-auc) so the direction of the association (e.g.
    presence vs. absence of a condition) doesn't penalize the score."""
    n1 = y.sum()
    n0 = len(y) - n1
    if n1 == 0 or n0 == 0:
        return pd.Series(0.5, index=X.columns)
    ranks = stats.rankdata(X.values, axis=0, method="average")
    sum_ranks_pos = ranks[y == 1].sum(axis=0)
    auc = (sum_ranks_pos - n1 * (n1 + 1) / 2) / (n1 * n0)
    auc = np.maximum(auc, 1 - auc)
    return pd.Series(auc, index=X.columns)


def kruskal_scores(X, y):
    """Kruskal-Wallis H statistic per feature, computed directly from ranks
    (equivalent to scipy.stats.kruskal but vectorized across every column):
        H = 12/(N(N+1)) * sum_i(R_i^2/n_i) - 3(N+1), tie-corrected.
    """
    N = len(y)
    ranks = stats.rankdata(X.values, axis=0, method="average")
    n1 = y.sum()
    n0 = N - n1
    R1 = ranks[y == 1].sum(axis=0)
    R0 = ranks[y == 0].sum(axis=0)
    H = (12 / (N * (N + 1))) * ((R1 ** 2) / n1 + (R0 ** 2) / n0) - 3 * (N + 1)

    # Tie correction: C = 1 - sum(t^3 - t) / (N^3 - N), computed per column
    # from the multiplicity of each repeated value (binary metadata has
    # heavy ties -- almost every column is just 0s and 1s).
    tie_correction = np.ones(X.shape[1])
    X_vals = X.values
    for i in range(X.shape[1]):
        _, counts = np.unique(X_vals[:, i], return_counts=True)
        tie_sum = np.sum(counts ** 3 - counts)
        tie_correction[i] = 1 - tie_sum / (N ** 3 - N)

    with np.errstate(divide="ignore", invalid="ignore"):
        H_corrected = np.where(tie_correction > 0, H / tie_correction, 0.0)
    H_corrected = np.nan_to_num(H_corrected, nan=0.0, posinf=0.0, neginf=0.0)
    return pd.Series(H_corrected, index=X.columns)

In [33]:
def run_pipeline(X, y, n_iterations, sample_fraction, top_k, seed):
    """Run the 1000-iteration resampling + scoring loop.
    Returns a DataFrame of selection frequency (0-1) per feature per method.
    """
    counts = {
        "anova": pd.Series(0, index=X.columns, dtype=int),
        "auc": pd.Series(0, index=X.columns, dtype=int),
        "kruskal": pd.Series(0, index=X.columns, dtype=int),
    }

    for it in range(n_iterations):
        # Stratified 80% resample: preserves the control:AMD ratio in
        # every iteration so small-class features aren't starved.
        X_sub, _, y_sub, _ = train_test_split(
            X, y,
            train_size=sample_fraction,
            stratify=y,
            random_state=RANDOM_SEED + it,
        )

        scores = {
            "anova": anova_scores(X_sub, y_sub),
            "auc": auc_scores(X_sub, y_sub),
            "kruskal": kruskal_scores(X_sub, y_sub),
        }

        for method, s in scores.items():
            top_features = s.sort_values(ascending=False).head(top_k).index
            counts[method].loc[top_features] += 1

        if (it + 1) % 100 == 0:
            print(f"  iteration {it + 1}/{n_iterations} done")

    freq = pd.DataFrame({m: c / n_iterations for m, c in counts.items()})
    return freq

In [34]:
def select_consistent_features(freq_df, top_n):
    """Take the top-N features per method (by selection frequency), then
    intersect across all three methods."""
    top_sets = {}
    for method in freq_df.columns:
        top_sets[method] = set(
            freq_df[method].sort_values(ascending=False).head(top_n).index
        )
    consistent = top_sets["anova"] & top_sets["auc"] & top_sets["kruskal"]
    return consistent, top_sets

In [35]:
freq_df = run_pipeline(X, y, N_ITERATIONS, SAMPLE_FRACTION, top_k, RANDOM_SEED)

consistent, top_sets = select_consistent_features(freq_df, top_n)

print(f"\nFeatures selected by all 3 methods: {len(consistent)}")
for f in sorted(consistent):
    print(f"  {f}  (anova={freq_df.loc[f,'anova']:.2f}, "
            f"auc={freq_df.loc[f,'auc']:.2f}, kruskal={freq_df.loc[f,'kruskal']:.2f})")

freq_df.to_csv("metadata_feature_selection_frequencies.csv")
pd.Series(sorted(consistent), name="feature").to_csv("metadata_selected_features.csv", index=False)
print("\nSaved: metadata_feature_selection_frequencies.csv, metadata_selected_features.csv")

  iteration 100/1000 done
  iteration 200/1000 done
  iteration 300/1000 done
  iteration 400/1000 done
  iteration 500/1000 done
  iteration 600/1000 done
  iteration 700/1000 done
  iteration 800/1000 done
  iteration 900/1000 done
  iteration 1000/1000 done

Features selected by all 3 methods: 17
  A69S_GG  (anova=1.00, auc=1.00, kruskal=1.00)
  A69S_TT  (anova=0.99, auc=1.00, kruskal=0.99)
  Y402H_CC  (anova=0.95, auc=1.00, kruskal=0.95)
  Y402H_TT  (anova=0.97, auc=1.00, kruskal=0.97)
  mh_Dementia  (anova=0.96, auc=0.87, kruskal=0.96)
  mh_asthma  (anova=0.88, auc=0.98, kruskal=0.88)
  mh_dementia  (anova=0.86, auc=0.99, kruskal=0.86)
  mh_fibromyalgia  (anova=0.84, auc=0.97, kruskal=0.84)
  mh_high chol  (anova=1.00, auc=1.00, kruskal=1.00)
  mh_hypothyroidism  (anova=0.96, auc=0.86, kruskal=0.96)
  mh_smoker (1 pk/day; 50yrs)  (anova=0.97, auc=0.94, kruskal=0.97)
  mh_smoker (half pk/day; 50+ yrs)  (anova=0.81, auc=0.77, kruskal=0.81)
  mh_smoking  (anova=1.00, auc=1.00, kruska